# Notebook 3: Fine-tuning the three selected models

Notebook 2 selected JobBERT, E5, and MPNet using the locked **mean-alias** representation. This notebook fine-tunes those three models with the same contrastive objective and comparable settings, then evaluates the fine-tuned models on the complete Task B development skill corpus.

Training uses positive pairs (`label > 0`), so contextual and core skills are both treated as relevant positives. The objective therefore fine-tunes retrieval relevance; it does not explicitly classify core versus contextual skills. Evaluation still uses graded qrels, where core is grade 2 and contextual is grade 1. JobBERT is a Router model, so the trainer maps the `anchor` and `positive` columns to its `query` and `document` routes.

Skill text in the training CSV is the processed canonical skill name. Evaluation uses the locked mean of all available validation aliases, and this representation choice is recorded in every output.

In [2]:
import ast
import gc
import json
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments, losses
from sentence_transformers.training_args import BatchSamplers

ROOT = Path.cwd()
if ROOT.name == "experiments":
    ROOT = ROOT.parent
if not (ROOT / "ranking_metrics.py").exists():
    raise FileNotFoundError(f"Repository root not found from {Path.cwd()}")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from config import RANDOM_SEED
from ranking_metrics import evaluate_query

TRAIN_PATH = ROOT / "data" / "processed" / "train.csv"
VAL_DIR = ROOT / "data" / "raw" / "TaskB" / "validation"
RESULTS_DIR = ROOT / "results"
MODELS_DIR = ROOT / "models"
RESULTS_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

SELECTED_PATH = RESULTS_DIR / "selected_models_for_finetuning.csv"
if not SELECTED_PATH.exists():
    raise FileNotFoundError("Run 05_locked_model_comparison.ipynb before this notebook")
selected = pd.read_csv(SELECTED_PATH)
if set(selected["model_short_name"]) != {"jobbert", "e5", "mpnet"}:
    raise ValueError("Expected JobBERT, E5, and MPNet in selected_models_for_finetuning.csv")
if set(selected["alias_strategy"]) != {"mean"}:
    raise ValueError("Fine-tuning must use the locked mean alias strategy")

EPOCHS = 2
BATCH_SIZE = 128
LEARNING_RATE = 2e-5
VAL_FRACTION = 0.05
SMOKE = False
print(f"Python: {sys.executable}")
print(f"Selected models: {', '.join(selected['model_short_name'])}")

Python: d:\apps\conda\envs\job-skill-matching\python.exe
Selected models: jobbert, e5, mpnet


C:\Users\kdani\AppData\Local\Temp\ipykernel_23668\2564018796.py:12: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments, losses
C:\Users\kdani\AppData\Local\Temp\ipykernel_23668\2564018796.py:13: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import BatchSamplers


## Prepare positive training pairs and fixed evaluation data

The validation split is by job title, using the same seed and fraction for every model. The official evaluation remains the raw Task B development set rather than the sampled negatives in `train.csv`.

In [3]:
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

train_all = pd.read_csv(TRAIN_PATH)
positives = train_all.loc[train_all["label"] > 0, ["job_name", "skill_name"]].drop_duplicates().reset_index(drop=True)
jobs = np.array(sorted(positives["job_name"].unique()))
rng = np.random.RandomState(RANDOM_SEED)
n_val_jobs = max(1, int(len(jobs) * VAL_FRACTION))
val_jobs = set(rng.choice(jobs, size=n_val_jobs, replace=False))
train_pairs = positives[~positives["job_name"].isin(val_jobs)].reset_index(drop=True)
validation_pairs = positives[positives["job_name"].isin(val_jobs)].reset_index(drop=True)
if SMOKE:
    train_pairs = train_pairs.sample(n=min(4000, len(train_pairs)), random_state=RANDOM_SEED).reset_index(drop=True)

train_dataset = Dataset.from_dict({
    "anchor": train_pairs["job_name"].astype(str).tolist(),
    "positive": train_pairs["skill_name"].astype(str).tolist(),
})
print(f"positive pairs: {len(positives):,}")
print(f"train pairs: {len(train_pairs):,}")
print(f"held-out training validation pairs: {len(validation_pairs):,} ({len(val_jobs)} jobs)")

queries = pd.read_csv(VAL_DIR / "queries", sep="\t")
corpus = pd.read_csv(VAL_DIR / "corpus_elements", sep="\t")
qrels = pd.read_csv(VAL_DIR / "qrels.tsv", sep="\t", header=None, names=["q_id", "iteration", "c_id", "relevance"])
corpus["skill_aliases"] = corpus["skill_aliases"].map(ast.literal_eval)
corpus["skill_aliases"] = corpus["skill_aliases"].map(lambda aliases: [str(alias).strip() for alias in aliases if str(alias).strip()])
skill_ids = corpus["c_id"].tolist()
skill_index = {skill_id: index for index, skill_id in enumerate(skill_ids)}
grades_by_job = {}
for job_id, job_qrels in qrels.groupby("q_id", sort=False):
    grades = np.zeros(len(skill_ids), dtype=np.float32)
    for row in job_qrels.itertuples(index=False):
        if row.c_id in skill_index:
            grades[skill_index[row.c_id]] = row.relevance
    grades_by_job[job_id] = grades
job_ids = [job_id for job_id in queries["q_id"] if job_id in grades_by_job]
query_texts = dict(zip(queries["q_id"], queries["jobtitle"]))
print(f"official evaluation queries: {len(job_ids)}")
print(f"official candidate skills: {len(skill_ids)}")

positive pairs: 114,686
train pairs: 108,870
held-out training validation pairs: 5,816 (150 jobs)
official evaluation queries: 304
official candidate skills: 9052


## Shared mean-alias evaluation

Every alias is encoded, the vectors for each skill are averaged, and the result is normalized. This is the same locked representation selected in Notebook 2.

In [4]:
def encode_normalized(model, texts, batch_size=64):
    return model.encode(texts, batch_size=batch_size, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)

def mean_alias_embeddings(model):
    aliases = corpus["skill_aliases"].tolist()
    counts = [len(items) for items in aliases]
    flattened = encode_normalized(model, [alias for items in aliases for alias in items])
    averaged = []
    start = 0
    for count in counts:
        vector = flattened[start : start + count].mean(axis=0)
        vector /= max(np.linalg.norm(vector), 1e-12)
        averaged.append(vector)
        start += count
    return np.asarray(averaged, dtype=np.float32)

def evaluate_model(model):
    job_embeddings = encode_normalized(model, [query_texts[job_id] for job_id in job_ids])
    skill_embeddings = mean_alias_embeddings(model)
    scores = job_embeddings @ skill_embeddings.T
    per_query = [evaluate_query(scores[index], grades_by_job[job_id]) for index, job_id in enumerate(job_ids)]
    return {name: float(np.mean([row[name] for row in per_query])) for name in per_query[0]}

def result_record(short_name, model_name, metrics, output_dir):
    return {
        "model_short_name": short_name,
        "model_name": model_name,
        "stage": "fine_tuned",
        "skill_representation": "mean",
        "loss": "MultipleNegativesRankingLoss",
        "positive_label_rule": "label > 0",
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "train_pairs": int(len(train_pairs)),
        "validation_jobs": int(len(val_jobs)),
        "n_queries": int(len(job_ids)),
        "n_candidate_skills": int(len(skill_ids)),
        "output_dir": str(output_dir) if output_dir else None,
        **{name: round(value, 6) for name, value in metrics.items()},
    }

## Fine-tune, evaluate, and save each selected model

Each fine-tuned model is saved under its own directory and evaluated with the same mean-alias protocol. Zero-shot scores are already reported by Notebook 2.

In [5]:
all_results = []
for row in selected.sort_values("rank").itertuples(index=False):
    short_name = row.model_short_name
    base_name = row.model_name
    output_dir = MODELS_DIR / f"{short_name}_mnrl_mean_alias"
    checkpoint_dir = MODELS_DIR / f"checkpoints_{short_name}"
    print(f"\n===== {short_name}: {base_name} =====")
    model = SentenceTransformer(base_name)
    epochs = 1 if SMOKE else EPOCHS
    loss = losses.MultipleNegativesRankingLoss(model)
    args = SentenceTransformerTrainingArguments(
        output_dir=str(checkpoint_dir),
        num_train_epochs=epochs,
        per_device_train_batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        warmup_steps=0.1,
        batch_sampler=BatchSamplers.NO_DUPLICATES,
        eval_strategy="no",
        save_strategy="no",
        logging_steps=50,
        seed=RANDOM_SEED,
        report_to="none",
        router_mapping={"anchor": "anchor", "positive": "positive"},
    )
    trainer = SentenceTransformerTrainer(model=model, args=args, train_dataset=train_dataset, loss=loss)
    trainer.train()
    if SMOKE:
        print("SMOKE=True: model was not saved")
        continue
    output_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(output_dir))
    fine_tuned_metrics = evaluate_model(model)
    print("fine-tuned:", {key: round(value, 4) for key, value in fine_tuned_metrics.items()})
    all_results.append(result_record(short_name, base_name, fine_tuned_metrics, output_dir))
    del trainer, loss, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if SMOKE:
    print("SMOKE=True: no final comparison was saved")
else:
    results_df = pd.DataFrame(all_results)
    results_df.to_csv(RESULTS_DIR / "finetuning_selected_models.csv", index=False)
    (RESULTS_DIR / "finetuning_selected_models.json").write_text(results_df.to_json(orient="records", indent=2), encoding="utf-8")
    display(results_df[["model_short_name", "stage", "ndcg_graded", "map_binary", "mrr_binary", "p@1", "p@5", "p@10"]])
    print("Saved: results/finetuning_selected_models.csv")
    print("Saved: results/finetuning_selected_models.json")


===== jobbert: TechWolf/JobBERT-v2 =====


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
50,4.931901
100,4.316520
150,3.548097
200,3.215114
250,3.035353
300,2.946460
350,2.818260
400,2.766958
450,2.725699
500,2.657167


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/1006 [00:00<?, ?it/s]

fine-tuned: {'ndcg_graded': 0.6981, 'ndcg_binary': 0.7438, 'map_binary': 0.2602, 'mrr_binary': 0.7071, 'p@1': 0.5625, 'p@5': 0.5033, 'p@10': 0.4766}


,model_short_name,stage,ndcg_graded,map_binary,mrr_binary,p@1,p@5,p@10
0,jobbert,fine_tuned,0.731888,0.325751,0.799661,0.667763,0.588158,0.550658
1,e5,fine_tuned,0.703989,0.269418,0.724304,0.565789,0.513816,0.503947
2,mpnet,fine_tuned,0.698082,0.260244,0.707061,0.562500,0.503289,0.476645


Saved: results/finetuning_selected_models.csv
Saved: results/finetuning_selected_models.json
